# Session 3 — Mountain Car: from tabular Q-learning to linear SARSA

**Reinforcement Learning atelier, Université Lumière Lyon 2**

By the end of this lab, you should be able to:
- implement tabular Q-learning in a discretised continuous environment;
- implement linear SARSA with function approximation;
- compare tabular and approximate methods;
- observe stability issues in TD learning with approximation.

This lab connects model-free control (Session 2) and value function approximation (today).
Part 5 then replaces the linear function by a neural network: the first step towards DQN.

| Part | Level | What |
|---|---|---|
| 0 | 🟢 | The Mountain Car environment |
| 1 | 🟢 | Tabular Q-learning with discretisation |
| 2 | 🟢 | Linear function approximation (SARSA) |
| 3 | 🟠 | Stability experiment |
| 4 | 🟢 | Comparison and discussion |
| 5 | 🟢 | From a linear function to a neural network (CartPole) |

Each `TODO` is followed by a **check cell**: if it runs without error, you are on the right track.
Questions are marked ❓. Cells marked **🔮 Predict** ask you to write down what you expect *before* running the experiment.

In [ ]:
# On Google Colab: uncomment the next line
# %pip install -q gymnasium

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

np.set_printoptions(precision=3, suppress=True)

---
## 🟢 Part 0 — The Mountain Car environment

- **State:** $s = (\text{position}, \text{velocity})$, with position $\in [-1.2, 0.6]$ and velocity $\in [-0.07, 0.07]$
- **Actions:** 0 push left, 1 no push, 2 push right
- **Reward:** $r = -1$ at every time step until the goal is reached
- **Goal:** reach position $\ge 0.5$

Gymnasium stops an episode after 200 steps. At the beginning the agent almost never reaches the goal that fast,
so it would never see the goal. We allow **1000 steps** per episode instead.

In [ ]:
env = gym.make("MountainCar-v0", max_episode_steps=1000)
LOW, HIGH = env.observation_space.low, env.observation_space.high
print("state space:", LOW, "to", HIGH)
print("actions:", env.action_space)

state, _ = env.reset(seed=0)
print("initial state (position, velocity):", state)

❓ **Conceptual questions**
1. Why can the car not directly drive to the goal?
2. Why is the reward always negative?
3. What behaviour must an optimal policy learn?

---
## 🟢 Part 1 — Tabular Q-learning with discretisation

The state space is continuous. We discretise it: position into 20 bins, velocity into 20 bins, so $20 \times 20 = 400$ states.

### TODO 1 — Discretisation

In [ ]:
def discretize_state(state, bins=20):
    # TODO 1: return (pos_bin, vel_bin), two integers in {0, ..., bins - 1}
    #         (scale position and velocity from [LOW, HIGH] to [0, bins), then clip)
    raise NotImplementedError

In [ ]:
# Check
assert discretize_state(LOW) == (0, 0)
assert discretize_state(HIGH) == (19, 19)                      # the upper bound goes into the last bin
assert discretize_state(np.array([-0.29, 0.001])) == (10, 10)
assert discretize_state(np.array([-0.29, 0.001]), bins=10) == (5, 5)
print("TODO 1 OK")

### TODO 2 — The Q-learning update

$$Q(s, a) \leftarrow Q(s, a) + \alpha \big( r + \gamma \max_{a'} Q(s', a') - Q(s, a) \big)$$

`Q` has shape `(bins, bins, 3)` and `s` is a pair `(pos_bin, vel_bin)`, so `Q[s]` holds the 3 action values of `s`.
When the episode is **terminated** (goal reached), there is no next state: the target is just $r$.

In [ ]:
def q_learning_update(Q, s, a, r, s_next, terminated, alpha=0.1, gamma=0.99):
    # TODO 2: update Q[s][a] in place
    raise NotImplementedError

In [ ]:
# Check
Q = np.zeros((2, 2, 3))
Q[1, 1] = [-4.0, -2.0, -3.0]
q_learning_update(Q, (0, 0), 2, -1.0, (1, 1), False, alpha=0.5, gamma=0.9)
assert np.isclose(Q[0, 0, 2], -1.4), Q[0, 0, 2]      # 0 + 0.5 * (-1 + 0.9 * (-2) - 0)
q_learning_update(Q, (0, 0), 2, -1.0, (1, 1), True, alpha=0.5, gamma=0.9)
assert np.isclose(Q[0, 0, 2], -1.2), Q[0, 0, 2]      # terminated: the target is -1
print("TODO 2 OK")

### Training

$Q(s, a) = 0$ at the start, $\alpha = 0.1$, $\gamma = 0.99$, $\varepsilon = 0.1$, 500 episodes.

In [ ]:
def epsilon_greedy(q_values, epsilon, rng):
    # Random action with probability epsilon, otherwise a best action (ties broken at random)
    if rng.random() < epsilon:
        return int(rng.integers(len(q_values)))
    return int(rng.choice(np.flatnonzero(q_values == q_values.max())))

def train_tabular(bins=20, n_episodes=500, alpha=0.1, gamma=0.99, epsilon=0.1, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((bins, bins, 3))
    lengths = []
    for episode in range(n_episodes):
        state, _ = env.reset(seed=seed * 10_000 + episode)
        s, done, t = discretize_state(state, bins), False, 0
        while not done:
            a = epsilon_greedy(Q[s], epsilon, rng)
            state, r, terminated, truncated, _ = env.step(a)
            s_next = discretize_state(state, bins)
            q_learning_update(Q, s, a, r, s_next, terminated, alpha, gamma)
            s, done, t = s_next, terminated or truncated, t + 1
        lengths.append(t)
    return np.array(lengths), Q

def moving_average(x, window=20):
    return np.convolve(x, np.ones(window) / window, mode="valid")

def plot_lengths(curves, title):
    plt.figure(figsize=(8, 4))
    for label, lengths in curves.items():
        line, = plt.plot(lengths, alpha=0.25)
        plt.plot(np.arange(19, len(lengths)), moving_average(lengths), color=line.get_color(), label=label)
    plt.xlabel("episode"); plt.ylabel("episode length (steps)"); plt.ylim(0, 1050)
    plt.title(title); plt.legend(); plt.show()

**🔮 Predict**: at the start every episode lasts 1000 steps (the car never reaches the goal). How long will an episode last after 500 episodes?

In [ ]:
lengths_tab, Q_tab = train_tabular(bins=20)      # about 5 seconds
plot_lengths({"tabular Q-learning, 20 bins": lengths_tab}, "Mountain Car: tabular Q-learning")
print("mean length of the last 50 episodes:", lengths_tab[-50:].mean())

❓ **Questions**
1. Does the agent learn?
2. What happens with 10 bins?
3. What happens with 50 bins?
4. Why does discretisation create problems?

In [ ]:
curves = {f"{b} bins": train_tabular(bins=b)[0] for b in [10, 20, 50]}    # about 15 seconds
plot_lengths(curves, "Mountain Car: number of bins")

---
## 🟢 Part 2 — Linear function approximation (SARSA)

We now approximate
$$\hat q(s, a; w) = w^\top x(s, a).$$

### Feature design
State features: $x(s) = [1, p, v, p^2, v^2, pv]$, where $p$ is the position and $v$ the velocity.
Action-dependent features put $x(s)$ in the block of the action:
$$
x(s, a) = \begin{cases}
[x(s), 0, 0] & a = 0 \\
[0, x(s), 0] & a = 1 \\
[0, 0, x(s)] & a = 2
\end{cases}
\qquad \text{Total dimension: } 18.
$$

### TODO 3 — The feature vector

In [ ]:
def features(state, action):
    # TODO 3: return x(s, a), a numpy array of length 18
    raise NotImplementedError

In [ ]:
# Check
x = features(np.array([0.5, -0.02]), 1)
assert x.shape == (18,)
assert np.allclose(x[6:12], [1.0, 0.5, -0.02, 0.25, 0.0004, -0.01]), x[6:12]
assert np.all(x[:6] == 0) and np.all(x[12:] == 0)
print("TODO 3 OK")

### TODO 4 — The SARSA update

$$\delta = r + \gamma\, \hat q(s', a') - \hat q(s, a), \qquad w \leftarrow w + \alpha\, \delta\, x(s, a)$$

With a linear function, $\hat q(s, a) = w^\top x(s, a)$ and its gradient is $x(s, a)$.
When the episode is terminated, there is no next state: $\delta = r - \hat q(s, a)$.

In [ ]:
def sarsa_update(w, x, r, x_next, terminated, alpha=0.01, gamma=0.99):
    # TODO 4: update w in place. x = x(s, a) and x_next = x(s', a')
    raise NotImplementedError

In [ ]:
# Check
w = np.zeros(18); w[1] = -2.0
x = np.zeros(18); x[0] = 1.0
x_next = np.zeros(18); x_next[1] = 1.0
sarsa_update(w, x, -1.0, x_next, False, alpha=0.5, gamma=0.9)
assert np.isclose(w[0], -1.4), w[0]        # delta = -1 + 0.9 * (-2) - 0
sarsa_update(w, x, -1.0, x_next, True, alpha=0.5, gamma=0.9)
assert np.isclose(w[0], -1.2), w[0]        # terminated: delta = -1 - (-1.4)
print("TODO 4 OK")

### Training

$w = 0$ at the start, $\alpha = 0.01$, $\gamma = 0.99$, $\varepsilon = 0.1$, 1000 episodes.

In [ ]:
def train_linear_sarsa(alpha=0.01, n_episodes=1000, gamma=0.99, epsilon=0.1, seed=0):
    rng = np.random.default_rng(seed)
    w = np.zeros(18)
    q_values = lambda state: np.array([w @ features(state, a) for a in range(3)])
    lengths = []
    for episode in range(n_episodes):
        state, _ = env.reset(seed=seed * 10_000 + episode)
        a, done, t = epsilon_greedy(q_values(state), epsilon, rng), False, 0
        while not done:
            next_state, r, terminated, truncated, _ = env.step(a)
            a_next = epsilon_greedy(q_values(next_state), epsilon, rng)
            sarsa_update(w, features(state, a), r, features(next_state, a_next), terminated, alpha, gamma)
            state, a, done, t = next_state, a_next, terminated or truncated, t + 1
        lengths.append(t)
    return np.array(lengths), w

In [ ]:
lengths_lin, w_lin = train_linear_sarsa(alpha=0.01)      # about 40 seconds
plot_lengths({"tabular Q-learning, 20 bins": lengths_tab, "linear SARSA, alpha = 0.01": lengths_lin},
             "Mountain Car: tabular vs linear")
print("mean length of the last 50 episodes:", lengths_lin[-50:].mean())

❓ Compare with tabular Q-learning.

---
## 🟠 Part 3 — Stability experiment

Repeat Part 2 with $\alpha = 0.01,\ 0.1,\ 0.5$ and plot all curves.

In [ ]:
runs = {0.01: (lengths_lin, w_lin)}                       # already trained in Part 2
for alpha in [0.1, 0.5]:
    runs[alpha] = train_linear_sarsa(alpha=alpha)          # about 1 minute in total
plot_lengths({f"alpha = {alpha}": lengths for alpha, (lengths, w) in runs.items()}, "Linear SARSA: step size")
for alpha, (lengths, w) in runs.items():
    print(f"alpha = {alpha}: last 50 episodes {lengths[-50:].mean():6.1f} steps, ||w|| = {np.linalg.norm(w):.1f}")

❓ **Questions**
1. For which $\alpha$ is learning stable?
2. Do you observe oscillations?
3. Why can large $\alpha$ cause divergence?
4. Why is tabular Q-learning more stable?

---
## 🟢 Part 4 — Comparison and discussion

❓ Answer:
1. Which method learns faster?
2. Which generalises better?
3. Which is more sensitive to hyperparameters?
4. Why is function approximation necessary in large problems?

---
## 🟢 Part 5 — From a linear function to a neural network (CartPole)

In Part 2 we chose the features $x(s, a)$ by hand. A **neural network** learns its own features:
it takes the state as input and outputs one value per action, $\hat q(s, \cdot\,; w)$.
The learning rule stays the same: move $\hat q(s, a; w)$ towards the Q-learning target $r + \gamma \max_{a'} \hat q(s', a'; w)$,
now with a gradient step on the squared error.

We use **CartPole**: push a cart left or right to keep a pole upright.
- **State:** 4 numbers (cart position, cart velocity, pole angle, pole angular velocity)
- **Actions:** 0 push left, 1 push right
- **Reward:** $+1$ for every step the pole stays up, at most 500 steps

Why not Mountain Car? There the agent first has to find the goal by chance, and a small network often never does.
In CartPole the reward comes at every step, so we quickly see what the network learns.

In [ ]:
import torch
import torch.nn as nn

cartpole = gym.make("CartPole-v1")

def make_q_network(n_inputs=4, n_actions=2):
    return nn.Sequential(nn.Linear(n_inputs, 64), nn.ReLU(),
                         nn.Linear(64, 64), nn.ReLU(),
                         nn.Linear(64, n_actions))

net = make_q_network()
print(net)
print("Q-values of a state:", net(torch.zeros(4)))

### TODO 5 — The Q-learning loss

$$L(w) = \big( \hat q(s, a; w) - y \big)^2, \qquad y = r + \gamma \max_{a'} \hat q(s', a'; w) \quad (y = r \text{ if terminated})$$

The target $y$ is treated as a **fixed number**: compute it inside `with torch.no_grad():`,
so that the gradient only moves $\hat q(s, a; w)$, as in the Q-learning update of Part 1.
`net(state)` returns the values of all actions; the inputs are numpy arrays, convert them with
`torch.as_tensor(x, dtype=torch.float32)`.

In [ ]:
def q_learning_loss(net, state, action, reward, next_state, terminated, gamma=0.99):
    # TODO 5: return the squared error between q_hat(state, action) and the target (a torch scalar)
    raise NotImplementedError

In [ ]:
# Check with a tiny linear "network" whose Q-values are the state itself
test_net = nn.Linear(2, 2, bias=False)
with torch.no_grad():
    test_net.weight.copy_(torch.eye(2))
loss = q_learning_loss(test_net, np.array([1.0, 2.0]), 0, 1.0, np.array([3.0, 5.0]), False, gamma=0.5)
assert np.isclose(loss.item(), 6.25), loss.item()        # target 1 + 0.5 * 5 = 3.5, q = 1
loss.backward()
assert torch.all(test_net.weight.grad[1] == 0), "the target must not receive gradients: use torch.no_grad()"
loss = q_learning_loss(test_net, np.array([1.0, 2.0]), 0, 1.0, np.array([3.0, 5.0]), True, gamma=0.5)
assert np.isclose(loss.item(), 0.0), loss.item()         # terminated: target = reward = 1 = q
print("TODO 5 OK")

### Training

After every step: compute the loss of this single transition and take one gradient step (Adam, learning rate $10^{-3}$).
$\gamma = 0.99$, $\varepsilon = 0.1$, 400 episodes.

In [ ]:
def train_neural_q(n_episodes=400, gamma=0.99, epsilon=0.1, lr=1e-3, seed=0):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    net = make_q_network()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    returns = []
    for episode in range(n_episodes):
        state, _ = cartpole.reset(seed=seed * 10_000 + episode)
        done, total = False, 0.0
        while not done:
            with torch.no_grad():
                q_values = net(torch.as_tensor(state, dtype=torch.float32)).numpy()
            action = epsilon_greedy(q_values, epsilon, rng)
            next_state, reward, terminated, truncated, _ = cartpole.step(action)
            loss = q_learning_loss(net, state, action, reward, next_state, terminated, gamma)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            state, done, total = next_state, terminated or truncated, total + reward
        returns.append(total)
    return np.array(returns), net

def plot_returns(curves, title):
    plt.figure(figsize=(8, 4))
    for label, returns in curves.items():
        line, = plt.plot(returns, alpha=0.25)
        plt.plot(np.arange(19, len(returns)), moving_average(returns), color=line.get_color(), label=label)
    plt.xlabel("episode"); plt.ylabel("return (steps with the pole up)"); plt.ylim(0, 520)
    plt.title(title); plt.legend(); plt.show()

**🔮 Predict**: will the return increase smoothly, episode after episode?

In [ ]:
returns_nn, net = train_neural_q(seed=0)      # about 20 seconds
plot_returns({"seed 0": returns_nn}, "CartPole: Q-learning with a neural network")
print("mean return of the last 50 episodes:", returns_nn[-50:].mean())

Run the same training with two other seeds:

In [ ]:
curves = {"seed 0": returns_nn}
for seed in [1, 2]:
    curves[f"seed {seed}"] = train_neural_q(seed=seed)[0]      # about 40 seconds
plot_returns(curves, "CartPole: three runs")

❓ **Questions**
1. Compare `q_learning_loss` with `q_learning_update` of Part 1. What is the same? What is different?
2. Is the learning curve smooth? Do the three seeds give the same result?
3. After a gradient step, the weights $w$ change. What happens to the target $y$ of the next update?
4. Two consecutive transitions are almost identical (the pole moved a little). Why is it a problem to train the network on them one after the other?

These two problems (a moving target and correlated samples) are what **DQN** fixes, with a **target network** and **experience replay**: that is the homework.